# Introduction to Agentic AI — ReAct Lab (Learner Version)

In this lab, you will compare a one-shot Direct response with a ReAct agent that can verify a plan, read feedback, revise its next action, and calculate a final cost.

## Learning goals

- Follow a `Thought → Action → Observation` loop.
- Use a few-shot trajectory to demonstrate the action protocol.
- Turn verifier feedback into a useful Reflection.
- Recognize the role of tools, parsing, validation, state, and step limits in an agent harness.

Start with `USE_REAL_API = False`. API-key setup and submission instructions are in [`README_en.md`](./README_en.md).


## 1. Challenge: Repair a Campus Event Plan

A school is planning a workshop for 150 people. It requires wheelchair access and a projector, lasts two hours, and must finish by 18:00. The presenter can start only at 14:00 or 16:00. The budget is 1,400, and projector rental costs 250.

| Venue | Capacity | Accessible | Built-in projector | Available start time | Fee |
|---|---:|:---:|:---:|---|---:|
| Hall A | 120 | Yes | Yes | 14:00, 16:00 | 900 |
| Hall B | 180 | Yes | No | 14:00 | 1,000 |
| Hall C | 160 | No | Yes | 16:00 | 800 |
| Hall D | 200 | Yes | Yes | 15:00 | 1,300 |

The initial plan is:

```json
{"venue": "Hall C", "start": "16:00", "rent_projector": false}
```

The ReAct agent may use `VerifyPlan[JSON]`, `Calculate[expression]`, and `Finish[JSON]`. It must verify the plan and cost before finishing.


In [ ]:
import ast
import json
import operator
import os
import re
import urllib.error
import urllib.request
from getpass import getpass

MODEL = os.getenv("ZAI_MODEL", "glm-4-flash-250414")
USE_REAL_API = False      # False: deterministic offline run; True: call the Zhipu API
MAX_STEPS = 12

CONSTRAINTS = """150 attendees; wheelchair access required; projector required;
duration is 2 hours; finish by 18:00; presenter is available only at 14:00 or 16:00;
total cost must not exceed 1400.""".strip()

VENUES = {
    "Hall A": {"capacity": 120, "accessible": True,  "projector": True,  "slots": ["14:00", "16:00"], "fee": 900},
    "Hall B": {"capacity": 180, "accessible": True,  "projector": False, "slots": ["14:00"],          "fee": 1000},
    "Hall C": {"capacity": 160, "accessible": False, "projector": True,  "slots": ["16:00"],          "fee": 800},
    "Hall D": {"capacity": 200, "accessible": True,  "projector": True,  "slots": ["15:00"],          "fee": 1300},
}

PROJECTOR_RENTAL_FEE = 250
INITIAL_PLAN = {"venue": "Hall C", "start": "16:00", "rent_projector": False}
EXPECTED_PLAN = {"venue": "Hall B", "start": "14:00", "rent_projector": True, "total_cost": 1250}

TASK = f"""Repair the campus workshop plan. Constraints: {CONSTRAINTS}
Venue data: {json.dumps(VENUES)}
Initial plan: {json.dumps(INITIAL_PLAN)}
Projector rental fee: {PROJECTOR_RENTAL_FEE}.
Use VerifyPlan before Calculate. Finish only with venue, start, rent_projector, and total_cost."""

print("Model:", MODEL, "| Live API:", USE_REAL_API)


## 2. API Mode

Keep `USE_REAL_API = False` for the first run. This deterministic mode is free and produces the same trace for everyone.

After your offline run passes, you may set `USE_REAL_API = True`. The notebook reads `ZAI_API_KEY` from the environment and falls back to hidden `getpass()` input when necessary.

See `README.md`for:

- how to create a Zhipu API key;
- environment-variable commands for Windows and macOS;
- key-safety guidance and common setup problems.


In [ ]:
class ZhipuClient:
    endpoint = "https://open.bigmodel.cn/api/paas/v4/chat/completions"

    def __init__(self, api_key=None):
        self.api_key = api_key or os.getenv("ZAI_API_KEY")
        if not self.api_key:
            print("ZAI_API_KEY was not found. Enter a temporary classroom API key.")
            self.api_key = getpass("Zhipu API key (input is hidden): ")
        if not self.api_key:
            raise ValueError("Missing API key")

    def chat(self, messages, temperature=0.2, max_tokens=500):
        body = json.dumps({
            "model": MODEL,
            "messages": messages,
            "temperature": temperature,
            "max_tokens": max_tokens,
        }).encode("utf-8")
        request = urllib.request.Request(
            self.endpoint,
            data=body,
            method="POST",
            headers={
                "Authorization": f"Bearer {self.api_key}",
                "Content-Type": "application/json",
            },
        )
        try:
            with urllib.request.urlopen(request, timeout=60) as response:
                payload = json.loads(response.read().decode("utf-8"))
        except urllib.error.HTTPError as exc:
            detail = exc.read().decode("utf-8", errors="replace")
            raise RuntimeError(f"Zhipu API error {exc.code}: {detail}") from exc
        return payload["choices"][0]["message"]["content"]


class OfflineClient:
    """A deterministic client for classroom practice and grading."""

    def chat(self, messages, temperature=0.2, max_tokens=500):
        if "DIRECT_BASELINE" in messages[0]["content"]:
            return '{"venue":"Hall D","start":"15:00","rent_projector":false,"total_cost":1300}'

        task_index = max(i for i, message in enumerate(messages) if message.get("content") == TASK)
        active_messages = messages[task_index + 1:]
        observations = [
            message["content"] for message in active_messages
            if message["role"] == "user" and message["content"].startswith("Observation:")
        ]
        reflections = [
            message["content"] for message in active_messages
            if message["role"] == "user" and message["content"].startswith("Reflection:")
        ]

        if not active_messages:
            return ('Thought: I should verify the given plan before changing it.\n'
                    'Action: VerifyPlan[{"venue":"Hall C","start":"16:00","rent_projector":false}]')
        if len(reflections) == 1 and not observations:
            return ('Thought: Hall D fixes access and capacity; verify it.\n'
                    'Action: VerifyPlan[{"venue":"Hall D","start":"15:00","rent_projector":false}]')
        if len(reflections) >= 2 and not observations:
            return ('Thought: Hall B at 14:00 meets timing; rent a projector and verify.\n'
                    'Action: VerifyPlan[{"venue":"Hall B","start":"14:00","rent_projector":true}]')
        if observations and "VALID" in observations[-1] and "1250" not in observations[-1]:
            return ('Thought: The plan is valid; calculate venue plus projector rental.\n'
                    'Action: Calculate[1000+250]')
        if observations and "Observation: 1250" in observations[-1]:
            return ('Thought: The valid plan and total cost are verified.\n'
                    'Action: Finish[{"venue":"Hall B","start":"14:00",'
                    '"rent_projector":true,"total_cost":1250}]')
        return ('Thought: I need to verify the initial plan.\n'
                'Action: VerifyPlan[{"venue":"Hall C","start":"16:00","rent_projector":false}]')


client = ZhipuClient() if USE_REAL_API else OfflineClient()


## 3. Direct Baseline

The Direct baseline gets one model call, no tools, and no second attempt. Before running it, predict whether one response will satisfy every capacity, access, equipment, time, and budget constraint.


In [ ]:
DIRECT_SYSTEM = """DIRECT_BASELINE
Answer once. You have no calculator, code execution, tools, or second attempt.
Return the requested JSON and do not claim to have used a tool.""".strip()

direct_text = client.chat([
    {"role": "system", "content": DIRECT_SYSTEM},
    {"role": "user", "content": TASK},
])
print(direct_text)


## 4. Provided Tools

`VerifyPlan` checks capacity, wheelchair access, equipment, time, and budget, then returns specific violations. `Calculate` evaluates the final arithmetic expression. You do not need to implement either tool.


In [ ]:
BIN_OPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.FloorDiv: operator.floordiv,
    ast.Mod: operator.mod,
}


def safe_calculate(expression):
    expression = expression.strip().strip("`").replace("×", "*").replace("÷", "/")

    def visit(node):
        if isinstance(node, ast.Expression):
            return visit(node.body)
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in BIN_OPS:
            value = BIN_OPS[type(node.op)](visit(node.left), visit(node.right))
            if abs(value) > 10**15:
                raise ValueError("Intermediate result is too large")
            return value
        raise ValueError("Only numeric arithmetic is allowed")

    value = visit(ast.parse(expression, mode="eval"))
    return str(int(value) if isinstance(value, float) and value.is_integer() else value)


def verify_plan(plan):
    if not isinstance(plan, dict) or plan.get("venue") not in VENUES:
        return "INVALID | unknown venue or invalid JSON"

    venue = VENUES[plan["venue"]]
    start = str(plan.get("start", ""))
    rent_projector = plan.get("rent_projector") is True
    violations = []

    if venue["capacity"] < 150:
        violations.append("capacity below 150")
    if not venue["accessible"]:
        violations.append("wheelchair access required")
    if start not in venue["slots"]:
        violations.append("venue unavailable at selected time")
    if start not in ("14:00", "16:00"):
        violations.append("presenter unavailable at selected time")
    if start and start[:2].isdigit() and int(start[:2]) + 2 > 18:
        violations.append("event finishes after 18:00")
    if not venue["projector"] and not rent_projector:
        violations.append("projector required")

    cost = venue["fee"] + (PROJECTOR_RENTAL_FEE if rent_projector else 0)
    if cost > 1400:
        violations.append("budget exceeded")

    if violations:
        return "INVALID | " + "; ".join(violations)

    rental_fee = PROJECTOR_RENTAL_FEE if rent_projector else 0
    return f"VALID | venue_fee={venue['fee']} | projector_rental_fee={rental_fee}"


assert verify_plan(INITIAL_PLAN).startswith("INVALID")
assert verify_plan({"venue": "Hall B", "start": "14:00", "rent_projector": True}).startswith("VALID")
assert safe_calculate("1000+250") == "1250"
print("Verifier and calculator checks passed.")


## 5. Your Task: Complete the Reasoning Scaffold (15–20 minutes)

Complete three functions:

1. `build_reasoning_prompt()` — define how the model reasons from the latest Observation and formats one Action;
2. `build_few_shot_messages()` — demonstrate the complete ReAct protocol with a smaller planning example;
3. `build_reflection()` — convert verifier feedback into a specific correction for the next attempt.

The API client, parser, tools, JSON handling, loop, and step limit are already provided. Do not modify those parts unless your instructor asks you to.


In [ ]:
def build_reasoning_prompt(constraints):
    # TODO 1: Return a system-prompt string.
    # Hint: require a short Thought, verify the complete initial plan first,
    # include all three plan fields, and output exactly one Action per turn.
    pass


def build_few_shot_messages():
    # TODO 2: Return a list of messages for a smaller planning example.
    # Suggested example: Room X seats 50 people, Room Y seats 80, and the event
    # needs 60 seats. Demonstrate Verify, Observation, Reflection, Calculate,
    # and Finish without copying the campus-workshop answer.
    pass


def build_reflection(feedback, previous_plan):
    # TODO 3: Return a string beginning with "Reflection:".
    # Include the verifier feedback and previous plan. State what can be kept,
    # what must change, and what a complete next proposal must contain.
    pass


## 6. Provided ReAct Loop

Run the next cell after completing the three functions. It adds your few-shot messages, executes one Action at a time, returns each Observation to the model, and calls your Reflection function when a proposal is rejected.


In [ ]:
ACTION_RE = re.compile(
    r"^\s*Action\s*:\s*(VerifyPlan|Calculate|Finish)\s*\[(.*?)\]\s*$",
    re.I | re.M | re.S,
)


def extract_plan(text):
    decoder = json.JSONDecoder()
    for match in re.finditer(r"\{", text):
        try:
            value, _ = decoder.raw_decode(text[match.start():])
        except json.JSONDecodeError:
            continue
        if isinstance(value, dict) and "venue" in value:
            return value

    for candidate in re.findall(r"\{[^{}]{1,300}\}", text, re.S):
        try:
            value = ast.literal_eval(candidate)
        except (SyntaxError, ValueError):
            continue
        if isinstance(value, dict) and "venue" in value:
            return value
    return None


def run_react(client, task, max_steps=12):
    system_prompt = build_reasoning_prompt(CONSTRAINTS)
    few_shot = build_few_shot_messages()

    if not isinstance(system_prompt, str) or not system_prompt.strip():
        return {"answer": None, "trace": [], "passed": False, "reason": "complete_prompt_TODO"}
    if not isinstance(few_shot, list) or len(few_shot) < 6:
        return {"answer": None, "trace": [], "passed": False, "reason": "complete_few_shot_TODO"}

    messages = [
        {"role": "system", "content": system_prompt},
        *few_shot,
        {"role": "user", "content": task},
    ]
    trace = []
    valid_plan = None
    calculated_cost = None
    previous_plan = INITIAL_PLAN

    for step in range(1, max_steps + 1):
        model_text = client.chat(messages)
        messages.append({"role": "assistant", "content": model_text})

        match = ACTION_RE.search(model_text)
        action = (match.group(1).title(), match.group(2).strip()) if match else None
        if action is None:
            bare_plan = extract_plan(model_text)
            action = ("Finish", model_text) if bare_plan else None

        if action is None:
            observation = "Format error: use exactly one VerifyPlan[...], Calculate[...], or Finish[...]."

        elif action[0].lower() == "verifyplan":
            plan = extract_plan(action[1])
            previous_plan = plan or previous_plan
            observation = verify_plan(plan)
            if observation.startswith("INVALID"):
                reflection = build_reflection(observation, previous_plan)
                if not isinstance(reflection, str) or not reflection.startswith("Reflection:"):
                    return {
                        "answer": None,
                        "trace": trace,
                        "passed": False,
                        "reason": "complete_reflection_TODO",
                    }
                trace.append({
                    "step": step,
                    "model": model_text,
                    "action": action,
                    "observation": observation,
                    "reflection": reflection,
                })
                messages.append({"role": "user", "content": reflection})
                continue
            valid_plan = plan

        elif action[0].lower() == "calculate":
            try:
                observation = safe_calculate(action[1])
            except (SyntaxError, ValueError, ZeroDivisionError, OverflowError) as exc:
                observation = f"Calculation error: {exc}"
            if observation.isdigit():
                calculated_cost = int(observation)

        else:
            answer = extract_plan(action[1]) or extract_plan(model_text)
            verified_core = {"venue": "Hall B", "start": "14:00", "rent_projector": True}
            verified_plan_ok = isinstance(valid_plan, dict) and all(
                valid_plan.get(key) == value
                for key, value in verified_core.items()
            )
            process_ok = verified_plan_ok and calculated_cost == 1250
            answer_ok = answer == EXPECTED_PLAN
            if process_ok and answer_ok:
                trace.append({
                    "step": step,
                    "model": model_text,
                    "action": action,
                    "observation": None,
                })
                return {"answer": answer, "trace": trace, "passed": True, "reason": "finish"}

            feedback = (
                "Finish rejected: the plan must be verifier-approved and total_cost "
                "must equal the calculator result."
            )
            reflection = build_reflection(feedback, answer or previous_plan)
            if not isinstance(reflection, str) or not reflection.startswith("Reflection:"):
                return {
                    "answer": None,
                    "trace": trace,
                    "passed": False,
                    "reason": "complete_reflection_TODO",
                }
            trace.append({
                "step": step,
                "model": model_text,
                "action": action,
                "observation": feedback,
                "reflection": reflection,
            })
            messages.append({"role": "user", "content": reflection})
            continue

        trace.append({
            "step": step,
            "model": model_text,
            "action": action,
            "observation": observation,
        })
        messages.append({
            "role": "user",
            "content": f"Observation: {observation}\nContinue with exactly one Action.",
        })

    return {"answer": None, "trace": trace, "passed": False, "reason": "max_steps"}


In [ ]:
react_result = run_react(client, TASK, MAX_STEPS)

for item in react_result["trace"]:
    print(f"\n--- Step {item['step']} ---")
    print(item["model"])
    if item["observation"] is not None:
        print("Observation:", item["observation"])
    if item.get("reflection"):
        print(item["reflection"])

print("\nResult:", react_result)


## 7. Completion Check

Your offline run should show:

- the Direct baseline does not pass the final check;
- the initial plan is verified before it is changed;
- an invalid proposal produces specific feedback and a Reflection;
- a revised plan passes `VerifyPlan`;
- the final cost comes from `Calculate`;
- `Finish` returns a complete JSON object;
- the final ReAct result contains `passed=True`.

Live-model traces may use different candidates or step counts. They still need to use verifier feedback, verify the final plan, calculate the cost, and finish within `MAX_STEPS`.

Submit the completed `Introduction_ReAct_Learner.ipynb`. Restart the kernel and confirm that the file and its saved outputs contain no API key.


In [ ]:
direct_answer = extract_plan(direct_text)
print(f"Direct: answer={direct_answer!r}, pass={direct_answer == EXPECTED_PLAN}")
print(f"ReAct: answer={react_result['answer']!r}, pass={react_result['passed']}")

if react_result["passed"]:
    print("Complete: verifier feedback, few-shot guidance, and Reflection produced a verified result.")
else:
    print("Not complete: check the reasoning prompt, few-shot messages, and Reflection function.")
